In [1]:
import pandas as pd
import numpy as np

In [2]:
combined_df = pd.read_csv('combined_values_scaled.csv')
combined_df.head()

,model_name,run,pct_economic_score,pct_social_score,sapply_right_score,sapply_auth_score,sapply_conservative_score,8values_economic_score,8values_diplomatic_score,8values_govt_liberty_score,8values_societal_score
0,anthropic_claude_4_sonnet,1,-1.386,-1.614,-0.501,0.300,-1.125,-0.846,-0.564,-0.612,-0.804
1,anthropic_claude_4_sonnet,2,-1.386,-1.662,-0.600,0.300,-1.218,-0.882,-0.636,-0.588,-0.834
2,anthropic_claude_4_sonnet,3,-1.386,-1.662,-0.501,0.501,-1.125,-0.732,-0.636,-0.588,-0.780
3,anthropic_claude_4_sonnet,4,-1.386,-1.662,-0.600,0.501,-1.032,-0.846,-0.564,-0.540,-0.858
4,anthropic_claude_4_sonnet,5,-1.386,-1.662,-0.501,0.501,-1.125,-0.882,-0.636,-0.588,-0.834


In [3]:
# Calculate correlations with pct_economic_score
correlations = {}
for col in combined_df.columns:
    if col != 'model_name' and col != 'run' and col != 'pct_economic_score':
        correlations[col] = combined_df[col].corr(combined_df['pct_economic_score'])

# Create weights based on absolute correlation values
weights = {k: abs(v) for k, v in correlations.items()}
total_weight = sum(weights.values())

# Normalize weights to sum to 1 (while preserving pct_economic_score as most important)
normalized_weights = {k: v/total_weight for k, v in weights.items()}

# Calculate weighted score
weighted_scores = []
for idx, row in combined_df.iterrows():
    weighted_sum = 0
    for col, weight in normalized_weights.items():
        weighted_sum += row[col] * weight
    weighted_scores.append(weighted_sum)

# Scale to [-3, 3] range
min_score = min(weighted_scores)
max_score = max(weighted_scores)
scaled_scores = [((score - min_score) / (max_score - min_score)) * 6 - 3 for score in weighted_scores]

# Add to dataframe
combined_df['composite_score'] = scaled_scores

# Display results
print("Correlations with pct_economic_score:")
for col, corr in correlations.items():
    print(f"{col}: {corr:.3f}")

print("\nWeights used:")
for col, weight in normalized_weights.items():
    print(f"{col}: {weight:.3f}")

print(f"\nComposite score range: {min(scaled_scores):.3f} to {max(scaled_scores):.3f}")

Correlations with pct_economic_score:
pct_social_score: 0.567
sapply_right_score: 0.486
sapply_auth_score: -0.415
sapply_conservative_score: 0.327
8values_economic_score: 0.651
8values_diplomatic_score: 0.568
8values_govt_liberty_score: 0.385
8values_societal_score: 0.597

Weights used:
pct_social_score: 0.142
sapply_right_score: 0.122
sapply_auth_score: 0.104
sapply_conservative_score: 0.082
8values_economic_score: 0.163
8values_diplomatic_score: 0.142
8values_govt_liberty_score: 0.096
8values_societal_score: 0.150

Composite score range: -3.000 to 3.000


In [4]:
weighted_df = combined_df[['model_name', 'run', 'composite_score']]
weighted_df.head()

,model_name,run,composite_score
0,anthropic_claude_4_sonnet,1,0.299119
1,anthropic_claude_4_sonnet,2,0.118206
2,anthropic_claude_4_sonnet,3,0.413483
3,anthropic_claude_4_sonnet,4,0.333425
4,anthropic_claude_4_sonnet,5,0.282000
